# Non-uniqueness Example

This notebook demonstrates the non-uniqueness of isometry pursuit solutions, as described in the "Non-uniqueness" subsection of method.tex.

We consider the matrix:
$$X = \begin{bmatrix}
1 & \frac{1}{2} & \frac{-1}{2} \\
0 & \frac{\sqrt{3}}{2} & \frac{\sqrt{3}}{2}
\end{bmatrix}$$

This example shows that there exist isometry pursuit solutions of minimum norm that are not isometries, and that by adding the first feature, we can achieve the optimum.


In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from isometry_pursuit.algorithm import isometry_pursuit, group_basis_pursuit
from isometry_pursuit.transformation import exponential_transformation
from isometry_pursuit.loss import isometry_loss

np.random.seed(1234)


In [2]:
# import isometry_pursuit

In [ ]:
# Construct the matrix X from the non-uniqueness example
# X = [[1, 1/2, -1/2],
#      [0, sqrt(3)/2, sqrt(3)/2]]
X = np.array([
    [1, 0.5, -0.5],
    [0, np.sqrt(3)/2, np.sqrt(3)/2]
])

print("Matrix X:")
print(X)
print(f"\nShape: {X.shape}")
print(f"\nColumn norms:")
for i in range(X.shape[1]):
    print(f"  Column {i}: {np.linalg.norm(X[:, i]):.4f}")


Matrix X:
[[ 1.         0.5       -0.5      ]
 [ 0.         0.8660254  0.8660254]]

Shape: (2, 3)

Column norms:
  Column 0: 1.0000
  Column 1: 1.0000
  Column 2: 1.0000


In [6]:
# Construct the matrix X from the non-uniqueness example
# X = [[1, 1/2, -1/2],
#      [0, sqrt(3)/2, sqrt(3)/2]]
X = np.array([
    [1, 0.5, 0.5],
    [0, np.sqrt(3)/2,- np.sqrt(3)/2]
])

print("Matrix X:")
print(X)
print(f"\nShape: {X.shape}")
print(f"\nColumn norms:")
for i in range(X.shape[1]):
    print(f"  Column {i}: {np.linalg.norm(X[:, i]):.4f}")


Matrix X:
[[ 1.         0.5        0.5      ]
 [ 0.         0.8660254 -0.8660254]]

Shape: (2, 3)

Column norms:
  Column 0: 1.0000
  Column 1: 1.0000
  Column 2: 1.0000


In [9]:
# Check if columns 2 and 3 form an orthonormal basis
cols_2_3 = X[:, [1, 2]]
print("Columns 2 and 3:")
print(cols_2_3)
print(f"\nDot product of columns 2 and 3: {np.dot(X[:, 1], X[:, 2]):.6f}")
print(f"Norm of column 2: {np.linalg.norm(X[:, 1]):.6f}")
print(f"Norm of column 3: {np.linalg.norm(X[:, 2]):.6f}")

# Check if they're orthonormal
gram_matrix = cols_2_3.T @ cols_2_3
print(f"\nGram matrix (should be identity if orthonormal):")
print(gram_matrix)


Columns 2 and 3:
[[ 0.5        0.5      ]
 [ 0.8660254 -0.8660254]]

Dot product of columns 2 and 3: -0.500000
Norm of column 2: 1.000000
Norm of column 3: 1.000000

Gram matrix (should be identity if orthonormal):
[[ 1.  -0.5]
 [-0.5  1. ]]


In [10]:
# Run isometry pursuit on the matrix
result = isometry_pursuit(X, power=1.0)

beta = result["beta"]
selected_indices = result["basis_pursuit_indices"]

print("Isometry Pursuit Results:")
print(f"Selected indices: {selected_indices}")
print(f"\nBeta matrix (P x D):")
print(beta)
# print(f"\nRow norms of beta (||beta_{p.}||_2):")
for i in range(beta.shape[0]):
    norm = np.linalg.norm(beta[i, :])
    print(f"  Row {i}: {norm:.6f}")

print(f"\nObjective value (sum of row norms): {result['objective_value']:.6f}")


Isometry Pursuit Results:
Selected indices: [0 1 2]

Beta matrix (P x D):
[[ 0.66666667  0.        ]
 [ 0.33333333  0.57735027]
 [ 0.33333333 -0.57735027]]
  Row 0: 0.666667
  Row 1: 0.666667
  Row 2: 0.666667

Objective value (sum of row norms): 2.000000


/Users/samsonkoelle/miniconda3/envs/isometry_pursuit/lib/python3.11/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


In [11]:
# Verify the constraint: w(X) @ beta should equal I_D
X_transformed = exponential_transformation(X, power=1.0)
constraint_check = X_transformed @ beta

print("Constraint verification (w(X) @ beta should equal I_D):")
print(constraint_check)
print(f"\nError (should be close to zero): {np.linalg.norm(constraint_check - np.eye(2), 'fro'):.2e}")


Constraint verification (w(X) @ beta should equal I_D):
[[ 1.00000000e+00 -3.01869640e-13]
 [-1.21418997e-16  1.00000000e+00]]

Error (should be close to zero): 3.02e-13


In [ ]:
# According to the theory, the solution should have:
# - Columns 2 and 3: loading of 1/sqrt(3) ≈ 0.577350
# - Column 1: loading of 1/sqrt(3) ≈ 0.577350
# - Total objective: 2/3 + 2/3 + 2/3 = 2.0

theoretical_norm = 1.0 / np.sqrt(3)
theoretical_objective = 2.0

print("Theoretical solution:")
print(f"  Expected row norm for each selected column: {theoretical_norm:.6f}")
print(f"  Expected total objective: {theoretical_objective:.6f}")
print(f"\nActual solution:")
print(f"  Actual row norms: {[np.linalg.norm(beta[i, :]) for i in selected_indices]}")
print(f"  Actual total objective: {result['objective_value']:.6f}")
print(f"\nDifference: {abs(result['objective_value'] - theoretical_objective):.2e}")


In [ ]:
# Compare solution using only columns 2 and 3 vs solution with all three columns
print("Solution using only columns 2 and 3:")
X_23 = X[:, [1, 2]]
X_23_transformed = exponential_transformation(X_23, power=1.0)
beta_23, _, obj_23 = group_basis_pursuit(X_23_transformed)
print(f"  Beta shape: {beta_23.shape}")
print(f"  Beta:")
print(beta_23)
print(f"  Row norms: {[np.linalg.norm(beta_23[i, :]) for i in range(beta_23.shape[0])]}")
print(f"  Objective: {obj_23:.6f}")

print("\nSolution using all three columns:")
print(f"  Selected indices: {selected_indices}")
print(f"  Objective: {result['objective_value']:.6f}")

print(f"\nDifference in objectives: {abs(result['objective_value'] - obj_23):.6f}")


In [ ]:
# Visualize the matrix columns and the solution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: The original vectors
ax1 = axes[0]
colors = ['red', 'blue', 'green']
for i in range(X.shape[1]):
    ax1.quiver(0, 0, X[0, i], X[1, i], angles='xy', scale_units='xy', scale=1, 
               color=colors[i], width=0.01, label=f'Column {i}')
ax1.set_xlim(-1.5, 1.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
ax1.axvline(x=0, color='k', linestyle='--', linewidth=0.5)
ax1.set_xlabel('Coordinate 1')
ax1.set_ylabel('Coordinate 2')
ax1.set_title('Original Matrix Columns')
ax1.legend()

# Plot 2: Beta coefficients
ax2 = axes[1]
row_norms = [np.linalg.norm(beta[i, :]) for i in range(beta.shape[0])]
bars = ax2.bar(range(len(row_norms)), row_norms, color=['red', 'blue', 'green'])
ax2.axhline(y=1/np.sqrt(3), color='k', linestyle='--', linewidth=1, 
            label=f'Theoretical: 1/√3 ≈ {1/np.sqrt(3):.4f}')
ax2.set_xlabel('Column Index')
ax2.set_ylabel('||β_{p.}||_2')
ax2.set_title('Beta Row Norms (Solution Coefficients)')
ax2.set_xticks(range(len(row_norms)))
ax2.set_xticklabels([f'Col {i}' for i in range(len(row_norms))])
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# Check if the selected columns form an isometry
if len(selected_indices) == 2:
    selected_cols = X[:, selected_indices]
    gram = selected_cols.T @ selected_cols
    print(f"Selected columns: {selected_indices}")
    print(f"\nGram matrix:")
    print(gram)
    print(f"\nIs identity? {np.allclose(gram, np.eye(2))}")
    print(f"\nThis demonstrates that the solution may not be an isometry,")
    print(f"even though it achieves the minimum objective value.")
